# day-06-prompt-workbench — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [9]:
# ---- Solution 1 ----
EX6 = EXAMPLES + textwrap.dedent('''\
    Ticket: "Login page throws a 500 error since this morning, nobody on my team can get in."
    Output: {"category": "bug", "priority": "high", "needs_human": true}
    Ticket: "Can you add two-factor auth?"
    Output: {"category": "feature_request", "priority": "low", "needs_human": false}
    Ticket: "Where do I download my past invoices?"
    Output: {"category": "billing", "priority": "low", "needs_human": false}
''')
p11 = ("p11_fewshot6", "few-shot",
       f"You triage support tickets. Output JSON matching:\n{SCHEMA}\nExamples:\n{EX6}",
       'Ticket: "{ticket}"')
s11 = agg(run_prompt(*p11))[0]
print(f"S1: p11_fewshot6 score={s11['score']:.2f} tok_in={s11['tok_in']:.0f}  "
      f"vs p06_fewshot_rules score={[d for d in summary if d['prompt']=='p06_fewshot_rules'][0]['score']:.2f}")

S1: p11_fewshot6 score=0.96 tok_in=268  vs p06_fewshot_rules score=0.94


In [10]:
# ---- Solution 2 ----
def per_field(text, gold):
    obj = extract_json(text)
    if not obj: return dict(category=0, priority=0, needs_human=0, parsed=0)
    return dict(category=int(obj.get("category")==gold["category"]),
                priority=int(obj.get("priority")==gold["priority"]),
                needs_human=int(obj.get("needs_human")==gold["needs_human"]), parsed=1)

import collections
field_tbl = collections.defaultdict(lambda: collections.defaultdict(list))
for p in PROMPTS:
    for _ in range(N_REPEATS):
        for ticket, gold in GOLD:
            txt,_ = call_model(p[2], p[3].format(ticket=ticket))
            for k,v in per_field(txt, gold).items():
                field_tbl[p[0]][k].append(v)
print(f"{'prompt':26s} {'category':>9} {'priority':>9} {'needs_human':>12}")
for name in [p[0] for p in PROMPTS]:
    t = field_tbl[name]
    print(f"{name:26s} {np.mean(t['category']):9.2f} {np.mean(t['priority']):9.2f} {np.mean(t['needs_human']):12.2f}")

prompt                      category  priority  needs_human
p01_zero_bare                   0.50      0.38         0.45
p02_zero_json                   1.00      0.80         0.88
p03_zero_schema                 1.00      0.82         0.87
p04_zero_schema_rules           1.00      0.78         0.85
p05_fewshot                     1.00      1.00         0.80
p06_fewshot_rules               1.00      1.00         0.82
p07_cot                         1.00      0.73         1.00
p08_cot_rules                   1.00      0.83         1.00
p09_fewshot_cot                 1.00      1.00         1.00
p10_fewshot_cot_strict          1.00      1.00         0.80


In [11]:
# ---- Solution 4 ----
PRICE_IN, PRICE_OUT = 3/1e6, 15/1e6
VOL = 500_000
print(f"{'prompt':26s} {'$/month @500k':>14}")
for d in summary:
    cost = VOL * (d["tok_in"]*PRICE_IN + d["tok_out"]*PRICE_OUT)
    print(f"{d['prompt']:26s} {cost:14,.0f}")
print("\nS4: the few-shot+CoT prompts can be 4-6x the monthly bill of schema+rules for a few "
      "points of score. Whether that's 'off the table' depends on the value of those points.")

prompt                      $/month @500k
p09_fewshot_cot                       552
p10_fewshot_cot_strict                518
p05_fewshot                           386
p06_fewshot_rules                     444
p07_cot                               348
p08_cot_rules                         398
p04_zero_schema_rules                 300
p03_zero_schema                       234
p02_zero_json                         188
p01_zero_bare                         147

S4: the few-shot+CoT prompts can be 4-6x the monthly bill of schema+rules for a few points of score. Whether that's 'off the table' depends on the value of those points.


In [12]:
# ---- Solution 5 ----
bestname = summary[0]["prompt"]
bp = [p for p in PROMPTS if p[0]==bestname][0]
M = np.zeros((5,5), int)
ci = {c:i for i,c in enumerate(CATEGORIES)}
for _ in range(N_REPEATS*3):
    for ticket, gold in GOLD:
        obj = extract_json(call_model(bp[2], bp[3].format(ticket=ticket))[0])
        if obj and obj.get("category") in ci:
            M[ci[gold["category"]], ci[obj["category"]]] += 1
print("rows=true, cols=pred:", CATEGORIES)
print(M)

rows=true, cols=pred: ['billing', 'bug', 'feature_request', 'account', 'other']
[[30  0  0  0  0]
 [ 0 45  0  0  0]
 [ 0  0 30  0  0]
 [ 0  0  0 45  0]
 [ 0  0  0  0 30]]


### Solutions 3 & 6 (sketch)

**S3:** wrap `run_prompt` in `for temp in (0.0, 0.4, 0.8)`, pass `temperature=temp` to
`call_model`, and plot `np.std` of score across repeats per temperature. Expect variance to
rise with temperature; for an extraction task you almost always want `temperature=0`.

**S6:** e.g. *"Your billing page has a bug that overcharged me."* — billing vs bug. The strong
prompts (p09/p10) should at least return the **same** category every repeat (low variance),
even if you'd debate which is "right". Consistency is a legitimate selection criterion when
ground truth is genuinely ambiguous.

### Answer key
1. They need different fixes: format failures → tighten the schema instruction / use
   constrained decoding; field errors → better rules, examples, or a smarter model. A blended
   score hides which one is hurting you.
2. Real models are stochastic at any temperature > 0, and even at 0 across model versions /
   load. One run is an anecdote; you need a mean and a spread.
3. Compute the value of the 7-point gap (support-triage errors caught) against the cost delta
   at your real volume. Often the cheaper prompt wins; sometimes the 7 points are worth it.
   The point is you *decide with numbers*.
4. The gold set, `score_response`, and the runner — that's an eval harness already.
5. Whether the prompt still parses the same way (new models format differently), then re-run
   the full bake-off; prompts are not portable across model versions without re-measuring.
6. From typical runs: few-shot tightens output *format/style* and borderline category calls;
   the explicit *rules* text is what lifts priority and needs_human accuracy, because those
   depend on thresholds the model can't infer from 3 examples.